# Module 8.2 — HTTP, REST & Consuming APIs

### Python Mastery · Track 8: Real-World Python

A great deal of real-world Python talks to web services over HTTP: fetching data from APIs, sending data to them, and handling the realities of networks such as errors, pagination, and rate limits. This module covers HTTP fundamentals and the `requests` and `httpx` libraries, working against a small API that runs locally so every cell is reproducible.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Rather than depend on an external service (which may be unavailable or change), this notebook starts a small **mock API server on your own machine**, so the requests are genuine HTTP calls with predictable responses. Run the server cell in Part 1 before the cells that call it.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Describe HTTP methods, status codes, headers, and JSON bodies.
2. Make GET and POST requests with `requests` and read responses.
3. Send query parameters, headers, and authentication tokens.
4. Handle errors, timeouts, and retries robustly.
5. Page through a paginated API and use `httpx` (including async).

**Prerequisites:** Tracks 1 to 6, and (for the async part) Track 5.

---

## Part 1 · HTTP Fundamentals and a Local Mock API

**HTTP** is a request/response protocol. A client sends a request with a **method** (GET to read, POST to create, PUT/PATCH to update, DELETE to remove), a path, optional **headers** (metadata such as content type and authorisation), and sometimes a **body**. The server replies with a **status code** (200 OK, 201 Created, 404 Not Found, 500 Server Error, and so on), headers, and usually a body, commonly JSON.

To make every example reproducible, the cell below starts a small API on `localhost`. It serves a few endpoints we will call throughout the module. Run it once; it runs in the background.

In [ ]:
import threading, json, http.server, socketserver

# A small in-memory dataset the mock API will serve.
BOOKS = [
    {"id": 1, "title": "Dune", "author": "Herbert", "year": 1965},
    {"id": 2, "title": "Neuromancer", "author": "Gibson", "year": 1984},
    {"id": 3, "title": "Snow Crash", "author": "Stephenson", "year": 1992},
    {"id": 4, "title": "Hyperion", "author": "Simmons", "year": 1989},
    {"id": 5, "title": "The Left Hand of Darkness", "author": "Le Guin", "year": 1969},
]

class APIHandler(http.server.BaseHTTPRequestHandler):
    def _send(self, code, payload, headers=None):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        for k, v in (headers or {}).items():
            self.send_header(k, v)
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        from urllib.parse import urlparse, parse_qs
        parsed = urlparse(self.path)
        params = parse_qs(parsed.query)

        if parsed.path == "/books":
            # Optional pagination via ?page=N&size=M.
            page = int(params.get("page", ["1"])[0])
            size = int(params.get("size", ["2"])[0])
            start = (page - 1) * size
            chunk = BOOKS[start:start + size]
            self._send(200, {"page": page, "size": size, "total": len(BOOKS), "items": chunk})
        elif parsed.path.startswith("/books/"):
            book_id = int(parsed.path.rsplit("/", 1)[1])
            match = next((b for b in BOOKS if b["id"] == book_id), None)
            self._send(200, match) if match else self._send(404, {"error": "not found"})
        elif parsed.path == "/secret":
            # Requires an Authorization header.
            if self.headers.get("Authorization") == "Bearer demo-token":
                self._send(200, {"message": "access granted"})
            else:
                self._send(401, {"error": "missing or invalid token"})
        else:
            self._send(404, {"error": "unknown endpoint"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        data = json.loads(self.rfile.read(length)) if length else {}
        new = {"id": len(BOOKS) + 1, **data}
        self._send(201, {"created": new})       # 201 Created

    def log_message(self, *args):
        pass                                     # silence the default logging

# Start the server on an OS-assigned free port, in a background thread.
_httpd = socketserver.TCPServer(("127.0.0.1", 0), APIHandler)
PORT = _httpd.server_address[1]
threading.Thread(target=_httpd.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{PORT}"
print("mock API running at", BASE)

The handler above is also a compact example of the standard library's `http.server`, which is fine for tests and tiny tools. Real services use a framework (Module 8.4). Note the status codes it returns: 200 for a successful read, 201 for a created resource, 401 for missing authentication, and 404 for an unknown path.

## Part 2 · Making Requests with `requests`

The `requests` library is the most popular way to make HTTP calls in Python. `requests.get(url)` performs a GET and returns a response object whose `.status_code`, `.json()`, `.text`, and `.headers` give you everything you need. Always check the status before trusting the body.

In [ ]:
import requests

# A basic GET request to our local API.
response = requests.get(f"{BASE}/books/1")
print("status code:", response.status_code)
print("content type:", response.headers["Content-Type"])

# Parse the JSON body into Python data.
book = response.json()
print("book:", book["title"], "by", book["author"])

# A request for something that does not exist returns 404.
missing = requests.get(f"{BASE}/books/999")
print("missing status:", missing.status_code, "->", missing.json())

## Part 3 · Query Parameters, Headers, and Authentication

Most APIs accept **query parameters** (the `?key=value` part of a URL) to filter or paginate, and require **headers** for content negotiation and **authentication**. With `requests`, pass `params=` for the query string (it handles encoding) and `headers=` for headers. A common auth scheme is a **bearer token** sent in the `Authorization` header.

In [ ]:
import requests

# Query parameters via params= (requests builds the ?page=...&size=... string).
response = requests.get(f"{BASE}/books", params={"page": 1, "size": 3})
data = response.json()
print("requested URL:", response.url)
print(f"page {data['page']} of {data['total']} total; this page has {len(data['items'])} items")
for item in data["items"]:
    print("  -", item["title"])

In [ ]:
import requests

# Authentication: the /secret endpoint requires a bearer token.
no_auth = requests.get(f"{BASE}/secret")
print("without token:", no_auth.status_code, no_auth.json())

with_auth = requests.get(f"{BASE}/secret", headers={"Authorization": "Bearer demo-token"})
print("with token:   ", with_auth.status_code, with_auth.json())

In [ ]:
import requests

# A POST request sends a body; json= serialises a dict and sets the content type.
new_book = {"title": "Foundation", "author": "Asimov", "year": 1951}
response = requests.post(f"{BASE}/books", json=new_book)
print("status:", response.status_code, "(201 means created)")
print("server echoed:", response.json()["created"])

## Part 4 · Robustness: Status Checks, Timeouts, and Retries

Networks fail, so robust code prepares for it. Three habits matter: always set a **timeout** so a call cannot hang forever; check the status (or call `raise_for_status()` to turn an error response into an exception); and **retry** transient failures with increasing delays (exponential backoff, from Module 5.6). The example shows a careful request wrapper.

In [ ]:
import requests
import time

def fetch_with_retries(url, max_attempts=3, timeout=5):
    """GET a URL with a timeout, raising on HTTP errors, retrying transient failures."""
    delays_used = []
    for attempt in range(max_attempts):
        try:
            response = requests.get(url, timeout=timeout)   # always set a timeout
            response.raise_for_status()                      # 4xx/5xx -> HTTPError
            return response.json(), delays_used
        except requests.HTTPError as e:
            # A 404 is not transient; do not retry it.
            if response.status_code == 404:
                return {"error": "not found"}, delays_used
            raise
        except requests.RequestException:
            if attempt == max_attempts - 1:
                raise
            delay = 2 ** attempt          # 1, 2, 4, ... (we record rather than sleep long)
            delays_used.append(delay)
    return None, delays_used

data, delays = fetch_with_retries(f"{BASE}/books/2")
print("fetched:", data["title"])
print("retries needed:", len(delays))

missing, _ = fetch_with_retries(f"{BASE}/books/999")
print("missing handled gracefully:", missing)

## Part 5 · Pagination and `httpx` (including async)

APIs that return many items usually **paginate**: each response gives one page plus enough information to request the next. You loop until you have collected everything. The example pages through all books. Then we switch to **`httpx`**, a modern client with an API very close to `requests` but with first-class **async** support, letting you fetch many resources concurrently (Track 5).

In [ ]:
import requests

# Page through the API until all items are collected.
def fetch_all_books(base):
    items, page, size = [], 1, 2
    while True:
        data = requests.get(f"{base}/books", params={"page": page, "size": size}).json()
        items.extend(data["items"])
        if len(items) >= data["total"]:      # collected everything
            break
        page += 1
    return items

all_books = fetch_all_books(BASE)
print(f"collected all {len(all_books)} books across pages:")
for b in all_books:
    print("  -", b["title"])

In [ ]:
import httpx

# httpx looks almost identical to requests for synchronous calls.
response = httpx.get(f"{BASE}/books/3", timeout=5)
print("httpx sync:", response.status_code, response.json()["title"])

# A client object reuses the connection across calls, which is more efficient.
with httpx.Client(base_url=BASE, timeout=5) as client:
    print("via client:", client.get("/books/4").json()["title"])

In [ ]:
import httpx
import asyncio

# httpx supports async, so we can fetch several books concurrently (see Track 5).
async def fetch_book(client, book_id):
    response = await client.get(f"/books/{book_id}")
    return response.json()["title"]

async def fetch_many(base, ids):
    async with httpx.AsyncClient(base_url=base, timeout=5) as client:
        tasks = [fetch_book(client, i) for i in ids]
        return await asyncio.gather(*tasks)       # concurrent requests

# In a notebook we await directly (Track 5): asyncio.run would fail here.
titles = await fetch_many(BASE, [1, 2, 3, 4, 5])
print("fetched concurrently:", titles)

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A simple GET and JSON (Easy)

In [ ]:
import requests
r = requests.get(f"{BASE}/books/5")
print(r.status_code, r.json()["title"])

### Example 2 — Reading a status code branch (Easy)

In [ ]:
import requests
for book_id in [1, 999]:
    r = requests.get(f"{BASE}/books/{book_id}")
    if r.status_code == 200:
        print("found:", r.json()["title"])
    else:
        print(f"id {book_id}: status {r.status_code}")

### Example 3 — Query parameters for paging (Medium)

In [ ]:
import requests
r = requests.get(f"{BASE}/books", params={"page": 2, "size": 2})
data = r.json()
print(f"page {data['page']}:", [b["title"] for b in data["items"]])

### Example 4 — Authenticated request (Medium)

In [ ]:
import requests
headers = {"Authorization": "Bearer demo-token"}
r = requests.get(f"{BASE}/secret", headers=headers)
print(r.status_code, r.json())
# Without the header it would be 401 Unauthorized.

### Example 5 — POST then read back (Difficult)

In [ ]:
import requests

payload = {"title": "Children of Time", "author": "Tchaikovsky", "year": 2015}
created = requests.post(f"{BASE}/books", json=payload)
print("create status:", created.status_code)
new_record = created.json()["created"]
print("assigned id:", new_record["id"], "| title:", new_record["title"])

### Example 6 — Concurrent fetch with httpx async (Difficult)

In [ ]:
import httpx, asyncio, time

async def get_title(client, book_id):
    r = await client.get(f"/books/{book_id}")
    return r.json().get("title", "missing")

async def run(base):
    async with httpx.AsyncClient(base_url=base, timeout=5) as client:
        start = time.perf_counter()
        titles = await asyncio.gather(*(get_title(client, i) for i in range(1, 6)))
        elapsed = time.perf_counter() - start
        return titles, elapsed

titles, elapsed = await run(BASE)
print("titles:", titles)
print(f"five concurrent requests completed in about {elapsed:.3f}s")

---

## Exercises

These use the mock API started in Part 1; make sure that cell has been run.

**Exercise 1 (Easy).** Use `requests.get` to fetch the book with id 2 and print its author.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Fetch an id that does not exist (for example 1000) and print the status code and the error JSON.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Use `params=` to fetch page 1 with size 4 from `/books`, and print the titles on that page along with the reported total.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Call `/secret` once without a token and once with `Authorization: Bearer demo-token`, printing both status codes and bodies.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Using `httpx.AsyncClient`, fetch books with ids 1, 3, and 5 concurrently with `asyncio.gather`, and print their titles.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import requests
print(requests.get(f"{BASE}/books/2").json()["author"])

**Solution 2**

In [ ]:
import requests
r = requests.get(f"{BASE}/books/1000")
print(r.status_code, r.json())

**Solution 3**

In [ ]:
import requests
data = requests.get(f"{BASE}/books", params={"page": 1, "size": 4}).json()
print("total:", data["total"], "| titles:", [b["title"] for b in data["items"]])

**Solution 4**

In [ ]:
import requests
print("no token:", requests.get(f"{BASE}/secret").status_code,
      requests.get(f"{BASE}/secret").json())
ok = requests.get(f"{BASE}/secret", headers={"Authorization": "Bearer demo-token"})
print("with token:", ok.status_code, ok.json())

**Solution 5**

In [ ]:
import httpx, asyncio

async def fetch(client, i):
    return (await client.get(f"/books/{i}")).json()["title"]

async def run():
    async with httpx.AsyncClient(base_url=BASE, timeout=5) as client:
        return await asyncio.gather(*(fetch(client, i) for i in [1, 3, 5]))

print(await run())

---

## Summary and Key Points

- HTTP is request/response: a method (GET, POST, PUT, PATCH, DELETE), path, headers, and optional body go out; a status code (200, 201, 401, 404, 500), headers, and usually JSON come back.
- `requests.get`/`post` return a response with `.status_code`, `.json()`, `.text`, and `.headers`; always check the status before trusting the body.
- Pass query parameters with `params=` (encoding is handled), headers with `headers=`, authentication commonly as a bearer token in `Authorization`, and a JSON body with `json=`.
- Robust calls set a `timeout`, use `raise_for_status()` to surface HTTP errors, and retry transient failures with exponential backoff, while not retrying definitive errors like 404.
- Paginated APIs are consumed by looping until all items are collected; `httpx` mirrors `requests` and adds async support, enabling concurrent fetches with `asyncio.gather`.

### Next module: 8.3 — CLI Tools & Automation

The next module covers building command-line tools: parsing arguments with `argparse`, an orientation to `Click` and `Typer`, configuration and exit codes, and writing reusable automation scripts.